# Food Delivery ETA Prediction - Exploratory Data Analysis

**Project:** Food Delivery ETA Prediction  
**Part 1:** Dataset + Exploratory Data Analysis  
**Dataset:** `data/raw/Food_Delivery_Times.csv`  
**Target variable:** `Delivery_Time_min`

This notebook performs Part 1 exploratory data analysis only. It verifies the dataset structure, data quality, distributions, outliers, correlations, and initial feature assessment before any preprocessing or modeling work.


## 1. Introduction

The goal of this EDA is to understand the food delivery dataset as it exists in the raw CSV file. The analysis is intentionally limited to dataset inspection and exploratory findings. Missing-value imputation, encoding, scaling, feature engineering, model training, MLflow, DVC, APIs, apps, containers, and automation are left for later project parts.


## 2. Import Libraries


In [3]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print(matplotlib.__version__)

sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

ModuleNotFoundError: No module named 'seaborn'

## 3. Load Dataset


In [ ]:
DATA_PATH = Path("../data/raw/Food_Delivery_Times.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/Food_Delivery_Times.csv")

TARGET = "Delivery_Time_min"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


### First Five Rows


In [ ]:
df.head()


### Last Five Rows


In [ ]:
df.tail()


## 4. Dataset Overview

The dataset contains 1,000 orders and 9 columns. Each row represents one delivery-order record with distance, order conditions, courier experience, preparation time, and the observed delivery time in minutes.


In [ ]:
overview = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Target variable"],
    "Value": [df.shape[0], df.shape[1], TARGET],
})
overview


In [ ]:
print("Column names:")
for idx, col in enumerate(df.columns, start=1):
    print(f"{idx}. {col}")


## 5. Data Types

The dataset has five numerical columns and four categorical columns. `Order_ID` is numeric by storage type, but it is an identifier rather than a measurement feature.


In [ ]:
data_types = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notna().sum().values,
    "Missing Count": df.isna().sum().values,
})
data_types


In [ ]:
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
predictor_numerical = [col for col in numerical_features if col != TARGET]

print(f"Target variable: {TARGET}")
print(f"Numerical columns ({len(numerical_features)}): {numerical_features}")
print(f"Categorical columns ({len(categorical_features)}): {categorical_features}")
print(f"Numerical predictor columns ({len(predictor_numerical)}): {predictor_numerical}")
print("Problem type: regression, because the target is a continuous numeric delivery time.")


## 6. Statistical Summary


### Numerical Summary

Delivery times range from 8 to 153 minutes, with a mean of 56.73 minutes and a median of 55.50 minutes. Distance ranges from 0.59 km to 19.99 km, and preparation time ranges from 5 to 29 minutes.


In [ ]:
df[numerical_features].describe().T


### Skewness

Most numerical columns are approximately symmetric. `Delivery_Time_min` has mild positive skew, meaning a small number of long deliveries stretch the upper tail.


In [ ]:
skewness = df[numerical_features].skew().rename("Skewness").to_frame()
skewness["Interpretation"] = np.select(
    [skewness["Skewness"] > 1, skewness["Skewness"] < -1, skewness["Skewness"].abs() <= 0.5],
    ["Highly positively skewed", "Highly negatively skewed", "Approximately symmetric"],
    default="Moderately skewed",
)
skewness


### Categorical Summary

The categorical columns have low cardinality. `Weather`, `Traffic_Level`, and `Time_of_Day` each contain 30 missing values, while `Vehicle_Type` has no missing values.


In [ ]:
for col in categorical_features:
    summary = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="Count")
    )
    summary["Percentage"] = (summary["Count"] / len(df) * 100).round(2)
    print(f"
{col}: {df[col].nunique(dropna=True)} non-missing unique values")
    display(summary)


## 7. Missing-Value Analysis

The dataset has 120 missing cells overall. Missingness is limited to four columns: `Weather`, `Traffic_Level`, `Time_of_Day`, and `Courier_Experience_yrs`, each with 30 missing values or 3.00% of the dataset. No imputation is performed in this Part 1 notebook.


In [ ]:
missing_table = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isna().sum().values,
    "Missing %": (df.isna().mean().values * 100).round(2),
})
missing_table = missing_table.sort_values(["Missing Count", "Column"], ascending=[False, True]).reset_index(drop=True)
missing_table


In [ ]:
columns_with_missing = missing_table.loc[missing_table["Missing Count"] > 0, "Column"].tolist()
print(f"Total missing cells: {int(df.isna().sum().sum())}")
print(f"Columns with missing values: {columns_with_missing}")


## 8. Duplicate Analysis

There are no fully duplicated rows in the raw dataset.


In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_percentage = duplicate_count / len(df) * 100
print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


## 9. Target Analysis

`Delivery_Time_min` is a continuous target for a regression problem. The distribution is mildly right-skewed: the mean is slightly higher than the median, and 6 deliveries fall above the IQR upper bound of 116 minutes.


In [ ]:
target_summary = df[TARGET].describe().to_frame(name=TARGET)
target_summary.loc["skewness"] = df[TARGET].skew()
target_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df[TARGET], bins=30, kde=False, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].axvline(df[TARGET].mean(), color="#D62728", linestyle="--", label=f"Mean: {df[TARGET].mean():.1f}")
axes[0].axvline(df[TARGET].median(), color="#2CA02C", linestyle="--", label=f"Median: {df[TARGET].median():.1f}")
axes[0].set_title("Distribution of Delivery Time")
axes[0].set_xlabel("Delivery Time (minutes)")
axes[0].set_ylabel("Order Count")
axes[0].legend()

sns.kdeplot(df[TARGET], fill=True, ax=axes[1], color="#4C78A8")
axes[1].set_title("Density of Delivery Time")
axes[1].set_xlabel("Delivery Time (minutes)")
axes[1].set_ylabel("Density")

sns.boxplot(y=df[TARGET], ax=axes[2], color="#F58518")
axes[2].set_title("Delivery Time Outlier Check")
axes[2].set_xlabel("")
axes[2].set_ylabel("Delivery Time (minutes)")

plt.tight_layout()
plt.show()


## 10. Numerical Feature Analysis

Among numerical predictors, `Distance_km` and `Preparation_Time_min` have the clearest relationship with delivery time. `Courier_Experience_yrs` has a weak negative relationship with the target, which is directionally plausible but should be evaluated further. `Order_ID` is an identifier and should not be interpreted as a predictive measurement even though it is stored as an integer.


In [ ]:
print(f"Numerical predictor features: {predictor_numerical}")


In [ ]:
n_features = len(predictor_numerical)
n_cols = 2
n_rows = int(np.ceil(n_features / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(predictor_numerical):
    sns.histplot(df[col].dropna(), bins=30, kde=True, ax=axes[idx], color="#4C78A8", edgecolor="white")
    axes[idx].axvline(df[col].mean(), color="#D62728", linestyle="--", label=f"Mean: {df[col].mean():.1f}")
    axes[idx].axvline(df[col].median(), color="#2CA02C", linestyle="--", label=f"Median: {df[col].median():.1f}")
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    axes[idx].legend()

for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(predictor_numerical):
    sns.scatterplot(data=df, x=col, y=TARGET, ax=axes[idx], alpha=0.65, color="#4C78A8")
    axes[idx].set_title(f"{col} vs Delivery Time")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Delivery Time (minutes)")

for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()


## 11. Categorical Feature Analysis

`Weather` and `Traffic_Level` show meaningful differences in delivery-time medians and means. Snowy weather and high traffic are associated with longer delivery times in this dataset. `Vehicle_Type` differences are comparatively small, and `Time_of_Day` differences are also modest among non-missing categories.


In [ ]:
print(f"Categorical features: {categorical_features}")


In [ ]:
n_cat_features = len(categorical_features)
n_cols = 2
n_rows = int(np.ceil(n_cat_features / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(categorical_features):
    order = df[col].value_counts(dropna=False).index
    sns.countplot(data=df, x=col, order=order, ax=axes[idx], color="#4C78A8")
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Order Count")
    axes[idx].tick_params(axis="x", rotation=35)

for idx in range(n_cat_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(categorical_features):
    order = df.groupby(col, dropna=False)[TARGET].median().sort_values().index
    sns.boxplot(data=df, x=col, y=TARGET, order=order, ax=axes[idx], color="#F58518")
    axes[idx].set_title(f"{col} vs Delivery Time")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Delivery Time (minutes)")
    axes[idx].tick_params(axis="x", rotation=35)

for idx in range(n_cat_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
for col in categorical_features:
    grouped_stats = (
        df.groupby(col, dropna=False)[TARGET]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("mean", ascending=False)
    )
    print(f"
{col} grouped delivery-time statistics:")
    display(grouped_stats.round(2))


## 12. Outlier Analysis

Using the 1.5 IQR rule, the predictor columns do not contain IQR outliers. The target column has 6 high-end outliers, representing 0.60% of the dataset, with delivery times above 116 minutes. These rows should be investigated in Part 2, but no rows are removed in this EDA.


In [ ]:
outlier_rows = []
for col in numerical_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(mask.sum()),
        "Outlier %": mask.mean() * 100,
        "Minimum": df[col].min(),
        "Maximum": df[col].max(),
    })

outlier_df = pd.DataFrame(outlier_rows)
outlier_df.round(2)


In [ ]:
target_q1 = df[TARGET].quantile(0.25)
target_q3 = df[TARGET].quantile(0.75)
target_iqr = target_q3 - target_q1
target_upper_bound = target_q3 + 1.5 * target_iqr

target_outliers = df.loc[df[TARGET] > target_upper_bound].sort_values(TARGET, ascending=False)
target_outliers


## 13. Correlation Analysis

`Distance_km` has the strongest linear relationship with `Delivery_Time_min` at 0.781. `Preparation_Time_min` has a moderate positive relationship at 0.307. `Courier_Experience_yrs` has a weak negative relationship at -0.090. `Order_ID` has near-zero correlation and is an identifier, so its correlation should not be treated as meaningful. No pair of numerical predictor columns exceeds an absolute correlation of 0.70, so strong numerical multicollinearity is not evident from this check.


In [ ]:
correlation_matrix = df[numerical_features].corr()
correlation_matrix.round(3)


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    linewidths=0.8,
    cbar_kws={"label": "Pearson correlation"},
)
plt.title("Correlation Heatmap of Numerical Columns")
plt.xlabel("Numerical Columns")
plt.ylabel("Numerical Columns")
plt.tight_layout()
plt.show()


In [ ]:
target_correlation = correlation_matrix[TARGET].sort_values(ascending=False).rename("Correlation with Delivery_Time_min")
target_correlation.to_frame().round(3)


In [ ]:
high_correlation_threshold = 0.70
predictor_corr = correlation_matrix.loc[predictor_numerical, predictor_numerical]

high_corr_pairs = []
for i, feature_1 in enumerate(predictor_corr.columns):
    for feature_2 in predictor_corr.columns[:i]:
        corr_value = predictor_corr.loc[feature_1, feature_2]
        if abs(corr_value) > high_correlation_threshold:
            high_corr_pairs.append({
                "Feature 1": feature_1,
                "Feature 2": feature_2,
                "Correlation": corr_value,
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    display(high_corr_df.round(3))
else:
    print("No numerical predictor pairs have absolute correlation above 0.70.")


## 14. Feature Assessment

This assessment is based on data quality, data type, distribution, relationship with the target, and domain meaning. It does not permanently remove any feature in Part 1.


In [ ]:
feature_assessment = pd.DataFrame([
    {
        "Feature": "Distance_km",
        "Assessment": "Potentially useful",
        "Reason": "Strong positive relationship with delivery time and clear operational meaning: longer routes generally take longer.",
    },
    {
        "Feature": "Preparation_Time_min",
        "Assessment": "Potentially useful",
        "Reason": "Moderate positive relationship with delivery time; it directly contributes to total order completion time.",
    },
    {
        "Feature": "Weather",
        "Assessment": "Potentially useful",
        "Reason": "Delivery-time averages vary by weather; snowy, rainy, and foggy conditions are associated with longer deliveries than clear weather.",
    },
    {
        "Feature": "Traffic_Level",
        "Assessment": "Potentially useful",
        "Reason": "High traffic has the highest mean and median delivery time, matching domain expectations.",
    },
    {
        "Feature": "Courier_Experience_yrs",
        "Assessment": "Requires further investigation",
        "Reason": "Weak negative relationship with delivery time and 3.00% missing values; may still help when combined with other features.",
    },
    {
        "Feature": "Time_of_Day",
        "Assessment": "Requires further investigation",
        "Reason": "Non-missing categories have similar delivery-time medians, but missing entries have a higher mean and should be reviewed before preprocessing.",
    },
    {
        "Feature": "Vehicle_Type",
        "Assessment": "Requires further investigation",
        "Reason": "Group differences are small in this EDA; usefulness may depend on interactions with distance, traffic, or weather.",
    },
    {
        "Feature": "Order_ID",
        "Assessment": "Potentially irrelevant",
        "Reason": "Identifier column with no operational measurement meaning; near-zero target correlation should not be interpreted as predictive value.",
    },
    {
        "Feature": "Delivery_Time_min",
        "Assessment": "Target",
        "Reason": "Continuous target variable to predict in later modeling work.",
    },
])
feature_assessment


## 15. EDA Findings & Initial Observations

### Dataset Overview
- The raw CSV contains 1,000 rows and 9 columns.
- The target variable is `Delivery_Time_min`.
- Numerical columns are `Order_ID`, `Distance_km`, `Preparation_Time_min`, `Courier_Experience_yrs`, and `Delivery_Time_min`.
- Categorical columns are `Weather`, `Traffic_Level`, `Time_of_Day`, and `Vehicle_Type`.
- This is a regression problem because the target is a continuous delivery time in minutes.

### Data Quality
- The dataset has 120 missing cells overall.
- `Weather`, `Traffic_Level`, `Time_of_Day`, and `Courier_Experience_yrs` each have 30 missing values, equal to 3.00% per column.
- There are no fully duplicated rows.
- `Order_ID` is complete and unique-looking as an identifier, but it should not be treated as a meaningful delivery predictor without further evidence.

### Target Distribution
- `Delivery_Time_min` has a mean of 56.73 minutes and a median of 55.50 minutes.
- The target ranges from 8 to 153 minutes.
- The target is mildly positively skewed, with 6 high-end IQR outliers above 116 minutes.

### Numerical Feature Behavior
- `Distance_km` ranges from 0.59 to 19.99 km and has the strongest linear relationship with delivery time.
- `Preparation_Time_min` ranges from 5 to 29 minutes and has a moderate positive relationship with delivery time.
- `Courier_Experience_yrs` ranges from 0 to 9 years and has a weak negative relationship with delivery time.
- No numerical predictor column has IQR outliers.

### Categorical Feature Behavior
- `Weather` has five observed categories: Clear, Rainy, Foggy, Snowy, and Windy. Snowy deliveries have the highest mean delivery time.
- `Traffic_Level` has Low, Medium, and High categories. High traffic has the highest mean and median delivery time.
- `Time_of_Day` has Morning, Afternoon, Evening, and Night categories. Non-missing category medians are similar, while missing time-of-day rows have higher average delivery time and need review.
- `Vehicle_Type` has Bike, Scooter, and Car categories. Delivery-time differences across vehicle types are small in this EDA.

### Outliers
- IQR analysis finds no outliers in the numerical predictor columns.
- `Delivery_Time_min` has 6 high-end outliers, or 0.60% of the dataset. These records should be investigated before deciding whether to cap, transform, or keep them in Part 2.

### Correlation Findings
- `Distance_km` has a strong positive correlation with delivery time: 0.781.
- `Preparation_Time_min` has a moderate positive correlation with delivery time: 0.307.
- `Courier_Experience_yrs` has a weak negative correlation with delivery time: -0.090.
- `Order_ID` has near-zero correlation with delivery time: -0.037, and is best treated as an identifier.
- No pair of numerical predictors exceeds an absolute correlation of 0.70, so this EDA does not show strong numerical multicollinearity.

### Potentially Useful Features
- `Distance_km`, because longer delivery distances are strongly associated with longer delivery times.
- `Preparation_Time_min`, because food preparation time contributes directly to total delivery time.
- `Weather`, because adverse weather categories show higher delivery-time averages.
- `Traffic_Level`, because high traffic is associated with longer delivery times.

### Features Requiring Further Investigation
- `Courier_Experience_yrs`, because it has missing values and only a weak individual correlation, but may be useful with other variables.
- `Time_of_Day`, because observed category differences are modest, but missing values show a different delivery-time pattern.
- `Vehicle_Type`, because its standalone group differences are small and may depend on interactions with distance or traffic.

### Potentially Irrelevant Features
- `Order_ID` is likely not useful for modeling because it is an identifier, not a delivery condition or operational measurement.

### Initial Considerations for Part 2
- Investigate missing-value handling for `Weather`, `Traffic_Level`, `Time_of_Day`, and `Courier_Experience_yrs`.
- Decide how to handle the 6 high-end target outliers after checking whether they represent valid long deliveries or data issues.
- Encode categorical variables only in Part 2.
- Consider scaling numerical variables only in Part 2 if the selected algorithms require it.
- Consider interaction features such as distance with traffic or weather only in Part 2; none are created in this notebook.
